# LABSCAN AI
## Intelligent Medical Laboratory Document Understanding Platform

**Advanced portfolio prototype | Healthcare + Bioinformatics**

Run the single code cell below. In Google Colab, upload PDF/JPG/PNG laboratory reports when prompted. In local Jupyter, place files in `LABSCAN_AI/input` before running.

> **Safety:** Experimental OCR and rule-based extraction only. Not clinically validated; verify all extracted results against the original report. Use synthetic or appropriately de-identified documents.

In [ ]:
# =====================================================================
# LABSCAN AI — Intelligent Medical Laboratory Document Understanding
# Complete single-cell prototype for Google Colab / Jupyter
# =====================================================================

import os, sys, re, json, shutil, subprocess, importlib.util, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

# 1. Install dependencies
def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for package, module in {
    "opencv-python-headless": "cv2",
    "pytesseract": "pytesseract",
    "pandas": "pandas",
    "numpy": "numpy",
    "Pillow": "PIL",
    "pdf2image": "pdf2image",
    "matplotlib": "matplotlib"
}.items():
    if importlib.util.find_spec(module) is None:
        pip_install(package)

import cv2
import pytesseract
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pdf2image import convert_from_path
from pytesseract import Output
from IPython.display import display

# Install Tesseract and Poppler when apt is available
if shutil.which("tesseract") is None:
    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "tesseract-ocr", "poppler-utils"],
            check=True
        )
    else:
        raise RuntimeError("Install Tesseract OCR and add it to your system PATH.")

# 2. Project directories
BASE = Path("LABSCAN_AI")
INPUT_DIR = BASE / "input"
OUTPUT_DIR = BASE / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("LABSCAN AI | Medical Laboratory Document Understanding")
print("=" * 70)

# 3. Upload documents
uploaded_paths = []
try:
    from google.colab import files
    print("Upload PDF, JPG, JPEG or PNG laboratory reports:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        destination = INPUT_DIR / Path(filename).name
        destination.write_bytes(content)
        uploaded_paths.append(destination)
except ImportError:
    uploaded_paths = [
        p for p in INPUT_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in {".pdf", ".jpg", ".jpeg", ".png"}
    ]
    if not uploaded_paths:
        raise RuntimeError(
            f"No documents found. Put PDF/JPG/PNG files in {INPUT_DIR.resolve()}."
        )

if not uploaded_paths:
    raise RuntimeError("No documents uploaded.")

# 4. Medical test dictionary
MEDICAL_TESTS = {
    "Hemoglobin": ["hemoglobin", "haemoglobin", "hb"],
    "WBC": ["white blood cell", "total leukocyte count", "total leucocyte count", "wbc", "tlc"],
    "RBC": ["red blood cell", "erythrocyte count", "rbc"],
    "Platelets": ["platelet count", "platelets", "platelet"],
    "Hematocrit": ["hematocrit", "haematocrit", "packed cell volume", "pcv"],
    "MCV": ["mean corpuscular volume", "mcv"],
    "MCH": ["mean corpuscular hemoglobin", "mch"],
    "MCHC": ["mchc"],
    "Neutrophils": ["neutrophils", "neutrophil"],
    "Lymphocytes": ["lymphocytes", "lymphocyte"],
    "Eosinophils": ["eosinophils", "eosinophil"],
    "Monocytes": ["monocytes", "monocyte"],
    "Basophils": ["basophils", "basophil"],
    "Glucose": ["fasting blood sugar", "blood glucose", "blood sugar", "glucose", "fbs"],
    "HbA1c": ["hba1c", "glycated hemoglobin", "glycosylated hemoglobin"],
    "Total Cholesterol": ["total cholesterol", "cholesterol total"],
    "HDL": ["hdl cholesterol", "hdl"],
    "LDL": ["ldl cholesterol", "ldl"],
    "Triglycerides": ["triglycerides", "triglyceride"],
    "Creatinine": ["serum creatinine", "creatinine"],
    "Urea": ["blood urea", "urea"],
    "TSH": ["thyroid stimulating hormone", "tsh"],
    "T3": ["total t3", "t3"],
    "T4": ["total t4", "t4"],
    "ALT": ["alanine aminotransferase", "sgpt", "alt"],
    "AST": ["aspartate aminotransferase", "sgot", "ast"],
    "Bilirubin": ["total bilirubin", "bilirubin"],
    "Calcium": ["serum calcium", "calcium"],
    "Vitamin D": ["25-oh vitamin d", "vitamin d"],
    "Vitamin B12": ["vitamin b12", "cobalamin"]
}
ALIASES = sorted(
    [(alias, name) for name, aliases in MEDICAL_TESTS.items() for alias in aliases],
    key=lambda x: len(x[0]), reverse=True
)
UNIT_PATTERN = (
    r"mg/dL|mg/L|g/dL|g/L|mmol/L|umol/L|µmol/L|mIU/L|IU/L|U/L|"
    r"ng/mL|pg/mL|%|fL|pg|cells/uL|cells/µL|million/uL|thousand/uL|/uL|/µL"
)
NUMBER_PATTERN = r"(?<![A-Za-z])[-+]?\d[\d,]*(?:\.\d+)?"
RANGE_PATTERN = r"([-+]?\d+(?:\.\d+)?)\s*[-–]\s*([-+]?\d+(?:\.\d+)?)"

# 5. Image preprocessing
def preprocess_image(pil_image):
    rgb = np.array(pil_image.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if gray.shape[1] < 1600:
        scale = 1600 / gray.shape[1]
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    denoised = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)
    return cv2.adaptiveThreshold(
        enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 11
    )

# 6. OCR
def perform_ocr(image):
    config = "--oem 3 --psm 6"
    text = pytesseract.image_to_string(image, config=config)
    data = pytesseract.image_to_data(image, config=config, output_type=Output.DATAFRAME)
    data = data.dropna(subset=["text"]).copy()
    data["text"] = data["text"].astype(str).str.strip()
    data = data[data["text"] != ""]
    data["conf"] = pd.to_numeric(data["conf"], errors="coerce")
    return text, data[data["conf"] >= 0]

# 7. Medical test identification
def identify_test(line):
    normalized = re.sub(r"[^a-z0-9 ]", " ", line.lower())
    normalized = re.sub(r"\s+", " ", normalized).strip()
    for alias, test_name in ALIASES:
        pattern = r"(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])"
        if re.search(pattern, normalized):
            return test_name
    return None

# 8. Extract test, value, unit, range and status
def extract_medical_data(text, document, page):
    records = []
    for line_number, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if len(line) < 3:
            continue
        test_name = identify_test(line)
        if not test_name:
            continue

        range_match = re.search(RANGE_PATTERN, line)
        low = high = None
        if range_match:
            low = float(range_match.group(1))
            high = float(range_match.group(2))
            value_text = line[:range_match.start()] + " " + line[range_match.end():]
            candidates = re.findall(NUMBER_PATTERN, value_text)
        else:
            candidates = re.findall(NUMBER_PATTERN, line)

        if not candidates:
            continue
        try:
            value = float(candidates[0].replace(",", ""))
        except (ValueError, IndexError):
            continue

        unit_match = re.search(UNIT_PATTERN, line, re.IGNORECASE)
        unit = unit_match.group(0) if unit_match else None

        upper = line.upper()
        explicit_flag = (
            "HIGH" if re.search(r"\b(HIGH|ABOVE)\b", upper)
            else "LOW" if re.search(r"\b(LOW|BELOW)\b", upper)
            else None
        )

        if low is not None and high is not None and low <= high:
            status = "LOW" if value < low else "HIGH" if value > high else "WITHIN_REPORTED_RANGE"
        elif explicit_flag:
            status = explicit_flag
        else:
            status = "REVIEW_REQUIRED"

        records.append({
            "document": document,
            "page": page,
            "test_name": test_name,
            "value": value,
            "unit": unit,
            "reference_low": low,
            "reference_high": high,
            "status": status,
            "source_line": line,
            "source_line_number": line_number
        })
    return records

# 9. Process every uploaded document/page
all_records, all_text, all_images, ocr_confidences = [], {}, {}, []

for file_path in uploaded_paths:
    print(f"\nProcessing: {file_path.name}")
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        pages = convert_from_path(str(file_path), dpi=250)
    elif suffix in {".jpg", ".jpeg", ".png"}:
        pages = [Image.open(file_path)]
    else:
        print("Skipped unsupported file:", file_path.name)
        continue

    for page_number, page_image in enumerate(pages, start=1):
        print(f"  Processing page {page_number}...")
        processed = preprocess_image(page_image)
        text, ocr_data = perform_ocr(processed)
        page_key = f"{file_path.name}_page_{page_number}"
        all_text[page_key] = text
        all_images[page_key] = processed
        if not ocr_data.empty:
            ocr_confidences.extend(ocr_data["conf"].dropna().tolist())
        all_records.extend(extract_medical_data(text, file_path.name, page_number))

# 10. Confidence proxy and review flags
average_ocr_confidence = float(np.mean(ocr_confidences)) if ocr_confidences else 0.0
for record in all_records:
    # Document-level OCR quality proxy, not calibrated per-field confidence.
    record["document_ocr_confidence"] = round(average_ocr_confidence / 100, 3)
    record["review_required"] = (
        record["status"] == "REVIEW_REQUIRED"
        or average_ocr_confidence < 70
        or record["unit"] is None
    )

# 11. Save CSV, JSON and raw OCR text
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = OUTPUT_DIR / f"labscan_results_{timestamp}.csv"
json_path = OUTPUT_DIR / f"labscan_results_{timestamp}.json"
text_path = OUTPUT_DIR / f"labscan_ocr_text_{timestamp}.txt"

df = pd.DataFrame(all_records)
df.to_csv(csv_path, index=False)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_records, f, indent=4, ensure_ascii=False)
with open(text_path, "w", encoding="utf-8") as f:
    for page_name, page_text in all_text.items():
        f.write(f"\n\n===== {page_name} =====\n{page_text}")

# 12. Results dashboard
print("\n" + "=" * 70)
print("LABSCAN AI — EXTRACTION SUMMARY")
print("=" * 70)
print("Documents:", len(uploaded_paths))
print("Pages:", len(all_text))
print("Candidate results extracted:", len(df))
print("Average OCR word confidence:", round(average_ocr_confidence, 2), "%")

if not df.empty:
    print("\nEXTRACTED LABORATORY DATA")
    display(df)
    print("\nSTATUS SUMMARY")
    display(df["status"].value_counts().rename_axis("Status").reset_index(name="Count"))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    counts = df["status"].value_counts()
    axes[0].bar(counts.index, counts.values)
    axes[0].set_title("Extracted Result Status")
    axes[0].set_ylabel("Number of Results")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].bar(["Average OCR confidence"], [average_ocr_confidence])
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel("Confidence (%)")
    axes[1].set_title("Document OCR Quality")
    plt.tight_layout()
    plt.show()

    print("\nRECORDS REQUIRING HUMAN REVIEW")
    review_df = df[df["review_required"]]
    display(review_df) if not review_df.empty else print("No records automatically flagged.")
else:
    print("\nNo recognised test records extracted. Inspect raw OCR text below.")

# 13. Raw OCR text preview
print("\nRAW OCR TEXT PREVIEW")
for page_name, page_text in all_text.items():
    print(f"\n--- {page_name} ---\n{page_text[:2500]}")

# 14. Preprocessed page preview
for page_name, processed in all_images.items():
    plt.figure(figsize=(12, 7))
    plt.imshow(processed, cmap="gray")
    plt.title(f"Preprocessed: {page_name}")
    plt.axis("off")
    plt.show()

# 15. Download results in Colab
print("\nOUTPUT FILES")
print("CSV :", csv_path.resolve())
print("JSON:", json_path.resolve())
print("OCR :", text_path.resolve())

if "google.colab" in sys.modules:
    from google.colab import files
    for output_file in [csv_path, json_path, text_path]:
        files.download(str(output_file))

print("\nLABSCAN AI PROCESSING FINISHED.")
print(
    "Safety: This is an experimental OCR/data-extraction prototype, "
    "not a clinically validated diagnostic tool. Verify all extracted "
    "values against the source report."
)
